In [2]:
import pandas as pd
import os

# ==========================================
# [설정값]
# ==========================================
OUTPUT_FILE = 'concat_problem_algorithms_by_solutions.csv'

# 병합할 타겟 파일 이름 리스트 자동 생성
# 1. 원본
target_files = ['problem_algorithms_by_solutions.csv']
# 2. 첫 번째 복사본
target_files.append('problem_algorithms_by_solutions - 복사본.csv')
# 3. 두 번째 ~ 아홉 번째 복사본
for i in range(2, 10):
    target_files.append(f'problem_algorithms_by_solutions - 복사본 ({i}).csv')

# ==========================================
# [메인 파이프라인]
# ==========================================
print("🚀 [Step 1] 분산 처리된 CSV 파일 로드 중...")

dfs = []
loaded_files_count = 0

# 파일들을 순회하며 데이터프레임으로 읽어오기
for file_name in target_files:
    if os.path.exists(file_name):
        print(f"  -> 📥 읽기 완료: {file_name}")
        df = pd.read_csv(file_name)
        dfs.append(df)
        loaded_files_count += 1
    else:
        print(f"  -> ⚠️ 경고: '{file_name}' 파일이 존재하지 않아 건너뜁니다.")

if not dfs:
    print("\n❌ 병합할 파일이 하나도 존재하지 않습니다. 스크립트를 종료합니다.")
    exit()

print(f"\n✅ 총 {loaded_files_count}개의 파일을 성공적으로 메모리에 로드했습니다.")
print("🚀 [Step 2] 데이터 정제, 중복 제거 및 병합 중...")

# 1. 모든 데이터프레임을 하나로 병합 (인덱스 무시하고 새로 부여)
merged_df = pd.concat(dfs, ignore_index=True)
total_rows_before = len(merged_df)
print(f"  -> 병합된 초기 데이터 행 개수: {total_rows_before}개")

# 2. 오름차순 정렬 (1순위: problem_id, 2순위: solution_order)
# numeric 정렬이 완벽하게 되도록 안전하게 형변환
merged_df['problem_id'] = pd.to_numeric(merged_df['problem_id'], errors='coerce')
merged_df['solution_order'] = pd.to_numeric(merged_df['solution_order'], errors='coerce')

merged_df.sort_values(by=['problem_id', 'solution_order'], ascending=[True, True], inplace=True)

# [추가] 3. 알고리즘 열 동적 추출 및 중복 조합 제거
# 식별자 열(problem_id, solution_order, count)을 제외한 나머지 모든 알고리즘 열 이름 추출
algo_cols = [col for col in merged_df.columns if col not in ['problem_id', 'solution_order', 'count']]

# problem_id와 모든 알고리즘 사용 여부가 동일한 행들을 찾아서 중복 제거
# 이미 solution_order 오름차순으로 정렬해두었으므로, keep='first'를 주면 제일 낮은 번호만 남습니다.
merged_df.drop_duplicates(subset=['problem_id'] + algo_cols, keep='first', inplace=True)
total_rows_after_dedup = len(merged_df)
print(f"  -> 중복 조합 제거 완료: {total_rows_before - total_rows_after_dedup}개의 중복 행 삭제됨 (순서가 빠른 것 1개만 유지)")

# [추가] 4. 알고리즘 열의 FALSE 값들을 엑셀 빈칸을 위해 빈 문자열로 변경
# 판다스가 읽어들일 때의 자료형(bool, str, int)을 모두 대비하여 다양한 False 케이스 치환
replace_dict = {False: '', 'False': '', 'false': '', 0: '', '0': ''}
merged_df[algo_cols] = merged_df[algo_cols].replace(replace_dict)

# 정렬 및 정제 후 인덱스 깔끔하게 재설정
merged_df.reset_index(drop=True, inplace=True)

print("🚀 [Step 3] 최종 결과물 CSV 저장 중...")

# UTF-8-SIG 인코딩으로 저장 (한글 깨짐 방지 및 엑셀 호환성 확보)
merged_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')

print("\n" + "="*50)
print("🎉 작업 완료 보고서")
print("="*50)
print(f"✔️ 병합된 파일 개수 : {loaded_files_count} / 10 개")
print(f"✔️ 제거된 중복 행수 : {total_rows_before - total_rows_after_dedup} 개")
print(f"✔️ 최종 생성된 행수 : {len(merged_df)} 개")
print(f"✔️ 저장된 파일 이름 : {OUTPUT_FILE}")
print(f"✔️ 특이사항         : False 값 빈칸 처리 적용 완료")
print("="*50)

🚀 [Step 1] 분산 처리된 CSV 파일 로드 중...
  -> 📥 읽기 완료: problem_algorithms_by_solutions.csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본.csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (2).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (3).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (4).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (5).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (6).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (7).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (8).csv
  -> 📥 읽기 완료: problem_algorithms_by_solutions - 복사본 (9).csv

✅ 총 10개의 파일을 성공적으로 메모리에 로드했습니다.
🚀 [Step 2] 데이터 정제, 중복 제거 및 병합 중...
  -> 병합된 초기 데이터 행 개수: 156740개
  -> 중복 조합 제거 완료: 151934개의 중복 행 삭제됨 (순서가 빠른 것 1개만 유지)
🚀 [Step 3] 최종 결과물 CSV 저장 중...

🎉 작업 완료 보고서
✔️ 병합된 파일 개수 : 10 / 10 개
✔️ 제거된 중복 행수 : 151934 개
✔️ 최종 생성된 행수 : 4806 개
✔️ 저장된 파일 이름 : concat_problem_algorithms_by_solutions.csv
✔️ 특이사항         : False 값 빈칸 처리 적용 완료


In [ ]:
'''
특정 문제가 완전히 분류 안 된 경우만 찾기

In [3]:
import pandas as pd
import os

# ==========================================
# [설정값]
# ==========================================
INPUT_FILE = 'concat_problem_algorithms_by_solutions.csv'

# ==========================================
# [메인 파이프라인]
# ==========================================
print(f"🚀 [Step 1] '{INPUT_FILE}' 파일 로드 중...")

if not os.path.exists(INPUT_FILE):
    print(f"❌ 에러: '{INPUT_FILE}' 파일이 존재하지 않습니다.")
    exit()

df = pd.read_csv(INPUT_FILE)
print(f"✅ 총 {len(df)}개의 행(솔루션 데이터)을 로드했습니다.")

print("🚀 [Step 2] '모든 솔루션의 count가 0'인 problem_id 탐색 중...")

# 1. problem_id 기준으로 그룹화하여, 각 그룹 내 'count' 열의 합계(sum)를 구합니다.
# count는 항상 0 이상의 정수이므로, 합계가 0이라는 것은 그 그룹의 모든 행이 0이라는 의미와 완벽히 같습니다.
grouped_counts = df.groupby('problem_id')['count'].sum()

# 2. 합계가 0인 problem_id들만 필터링하여 리스트로 추출합니다.
unclassified_problem_ids = grouped_counts[grouped_counts == 0].index.tolist()

# ==========================================
# [결과 출력]
# ==========================================
print("\n" + "="*50)
print("📊 미분류(count=0) 알고리즘 문제 탐색 결과")
print("="*50)
print(f"✔️ 탐색된 전체 문제 종류 수: {len(grouped_counts)} 개")
print(f"🚨 모든 솔루션에서 알고리즘이 도출되지 않은 문제 수: {len(unclassified_problem_ids)} 개")

if len(unclassified_problem_ids) > 0:
    print("\n[상세 명단] 해당 problem_id 리스트:")
    # 리스트를 한 줄에 10개씩 깔끔하게 출력
    for i in range(0, len(unclassified_problem_ids), 10):
        chunk = unclassified_problem_ids[i:i+10]
        print("  " + ", ".join(map(str, chunk)))
print("="*50)

🚀 [Step 1] 'concat_problem_algorithms_by_solutions.csv' 파일 로드 중...
✅ 총 4806개의 행(솔루션 데이터)을 로드했습니다.
🚀 [Step 2] '모든 솔루션의 count가 0'인 problem_id 탐색 중...

📊 미분류(count=0) 알고리즘 문제 탐색 결과
✔️ 탐색된 전체 문제 종류 수: 3542 개
🚨 모든 솔루션에서 알고리즘이 도출되지 않은 문제 수: 382 개

[상세 명단] 해당 problem_id 리스트:
  194.0, 244.0, 269.0, 297.0, 310.0, 485.0, 539.0, 564.0, 568.0, 601.0
  627.0, 664.0, 702.0, 771.0, 851.0, 921.0, 993.0, 1057.0, 1249.0, 1286.0
  1545.0, 1575.0, 1593.0, 1594.0, 1712.0, 2012.0, 2135.0, 2140.0, 2146.0, 2184.0
  2239.0, 2318.0, 2323.0, 2412.0, 2436.0, 2547.0, 2834.0, 2858.0, 2975.0, 3024.0
  3030.0, 3034.0, 3123.0, 3352.0, 3484.0, 3633.0, 3699.0, 3865.0, 3893.0, 3924.0
  3978.0, 3986.0, 4004.0, 4027.0, 4036.0, 4134.0, 4178.0, 4237.0, 4429.0, 4493.0
  4525.0, 4612.0, 4739.0, 4820.0, 4827.0, 4935.0, 5014.0, 5075.0, 5100.0, 5140.0
  5191.0, 5337.0, 5358.0, 5366.0, 5376.0, 5379.0, 5386.0, 5481.0, 5484.0, 5534.0
  5605.0, 5621.0, 5628.0, 5694.0, 5816.0, 5851.0, 5972.0, 6003.0, 6068.0, 6096.0
  6131.0, 6189.0, 6